# Extraction of medications on admission from clinical notes

## Extraction run over the experiment grid

- Each note of the selected environment is processed once per `(model, strategy)` cell:
  3 models x 3 prompting strategies.
- All input variables (paths, prompt files, grid, Ollama runtime settings) are defined
  **here** and passed as parameters. `utils/llm/llm_extraction.py` holds pure logic only.
- The model sits on the **outer** loop, so it is loaded into VRAM once per cell instead
  of being swapped in and out.
- Notes inside a cell run concurrently: `ExtractionRunner` is frozen and stateless, and
  each call writes its own file, so threads never contend.
- Caching in `ExtractionRunner.extract()` makes this safely re-runnable after an
  interruption, **provided the run is resumed into the same directory**: set
  `RESUME_RUN` to the folder name of the interrupted run.

## Imports

In [ ]:
import json
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm import tqdm

from clinical_notes_extraction.config import PROJECT_ROOT
from clinical_notes_extraction.utils.llm.llm_extraction import ExtractionRunner

## Configuration

`ENVIRONMENT` selects the phase data:

- `dev` for model/strategy selection -- the winning combination is chosen on this data;
- `prod` only for the final one-shot run of that combination on the held-out test set.

`STRATEGIES` drives the whole notebook: which prompt shells are read, which example
assets are loaded and which grid cells are executed.

`MAX_WORKERS` must not exceed the server's `OLLAMA_NUM_PARALLEL`; above that, requests
just queue. It also multiplies KV-cache VRAM: the server allocates `num_ctx` per slot,
not per model.

`RESUME_RUN` is the resume switch. Left as `None`, a fresh timestamped run directory is
created. Set to an existing folder name (e.g. `"20260812_142530"`), the run continues
inside it: `extract()` finds the records already on disk and only fills in the gaps.
This is the whole point of the cache -- a new directory would silently redo everything.

In [ ]:
# Phase: "dev" for train, "prod" for the final held-out run.
ENVIRONMENT = "prod"

# Strategies under test.
STRATEGIES = [
    "few_shot"
]

# Concurrent notes per (model, strategy) cell.
MAX_WORKERS = 4

# Persist the fully assembled prompt of every call, under
# <run_dir>/<model>/prompts/<strategy>/<note_id>. One file per note is thread-safe, the
# same way the result files are, so this does not constrain MAX_WORKERS. Kept on by
# default: under the dynamic strategy the prompt depends on the medoid picked at runtime
# and cannot be reconstructed from the template afterwards.
SAVE_PROMPTS = True

# None = start a new run. Otherwise, the name of the run directory to resume into.
RESUME_RUN = None

ROOT = PROJECT_ROOT / "scripts" / "3_information_extraction" / "3_2_medications_on_admission"
CONFIG_DIR = PROJECT_ROOT / "config"
PROMPTS_DIR = ROOT / "prompts"
DATA_DIR = ROOT / "data"
RESULTS_DIR = DATA_DIR / "llm_extraction_results" / ENVIRONMENT

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## Load configuration and split

`runnable_models.json` carries the runnable model tags and the Ollama block:

```json
{
  "ollama": {
    "url": "http://localhost:11434",
    "timeout_seconds": 900,
    "keep_alive": "30m",
    "options": {
      "temperature": 0.0,
      "num_ctx": 50000,
      "num_predict": 8000,
      "seed": 42
    }
  },
  "models": ["medgemma:27b", "gemma3:27b", "llama4:scout"]
}
```

`num_ctx` is the shared budget for prompt **and** generation; `num_predict` caps the
generation alone, so a degenerate repetition loop is bounded instead of running until
the window is full. `ExtractionRunner.__post_init__` rejects a config missing either.

`models` is a flat list of Ollama tags: the same string is checked against `/api/tags`
and handed to `ExtractionRunner(model=...)`.

The split holds `note_id` and `text`; `embedding` is only read when the dynamic strategy
is active, since it is what selects the medoid example.

In [ ]:
config = json.loads((CONFIG_DIR / "runnable_models.json").read_text(encoding="utf-8"))

REQUIRED_OLLAMA_KEYS = {"url", "options", "timeout_seconds"}

try:
    models = config["models"]
    ollama_config = config["ollama"]
except KeyError as exc:
    raise KeyError(
        f"runnable_models.json is missing {exc}. Top-level keys found: {sorted(config)}"
    ) from exc

missing_keys = REQUIRED_OLLAMA_KEYS - ollama_config.keys()
if missing_keys:
    raise KeyError(f"'ollama' block is missing: {sorted(missing_keys)}")

# "embedding" is only needed by the dynamic strategy, which picks its example by
# similarity to the annotated medoids.
columns = ["note_id", "text"] + (["embedding"] if "dynamic" in STRATEGIES else [])

env_df = pd.read_parquet(DATA_DIR / f"sample/{ENVIRONMENT}_sample.parquet")
notes = env_df[columns].to_dict("records")

# note_id is the result filename: a blank or duplicated one silently overwrites another
# note's record, and the collision is invisible until the evaluation comes up short.
note_ids = [str(note["note_id"]).strip() for note in notes]
if not all(note_ids):
    raise ValueError("Blank note_id in the split; result files would collide")
if len(set(note_ids)) != len(note_ids):
    raise ValueError("Duplicate note_id in the split; result files would collide")

# TEMPORARY: smoke test before the full grid. Remove both lines afterwards.
# models = ["deepseek-r1:70b"]
# notes = notes[:3]

n_cells = len(models) * len(STRATEGIES) * len(notes)
print(f"{len(models)} models x {len(STRATEGIES)} strategies x {len(notes)} notes = {n_cells} cells")

## Load prompt assets once

Reading these here avoids re-reading the same files on every grid cell.

- `role.md` is the system prompt;
- the strategy templates are the user prompt shells, read only for the active strategies;
- `expected_template.md` is the shared output schema injected into every shell.

In [ ]:
# System prompt (persona) and the shared expected-output schema.
role = (PROMPTS_DIR / "role.md").read_text(encoding="utf-8")
expected_template = (PROMPTS_DIR / "expected_template.md").read_text(encoding="utf-8")

# One user-prompt shell per active strategy, keyed by strategy name.
strategy_templates = {
    strategy: (PROMPTS_DIR / f"{strategy}.md").read_text(encoding="utf-8")
    for strategy in STRATEGIES
}

## Example assets

Loaded only by the strategies that consume them, so a zero-shot run touches no example
file at all.

- **few-shot**: a fixed pool of curated examples, external to the 32-note sample;
- **dynamic**: one annotated medoid per cluster, selected per note at assembly time.

In [ ]:
# Static few-shot examples, written by hand in Markdown.
FEW_SHOT_EXAMPLES_PATH = DATA_DIR / "annotations/few_shot/few_shot.md"
FEW_SHOT_EXAMPLES = FEW_SHOT_EXAMPLES_PATH.read_text(encoding="utf-8").strip()

### Dynamic examples

The example shown to the model is the medoid whose embedding is closest to the note's,
measured by cosine similarity. This is the same metric used to pick the medoids
themselves, keeping the metric space consistent between example construction and example
selection.

The medoid bank -- ids, embedding matrix and rendered example blocks -- is built once, the
way a service would load it at start-up. Selection itself happens per note, at prompt
assembly time, so the code path is the same one a production call would take: receive a
note, embed it, pick the nearest medoid.

In [ ]:
def to_builtin(value):
    """Parquet structs deserialize into numpy containers, which json.dumps rejects."""
    if isinstance(value, np.ndarray):
        return [to_builtin(item) for item in value]
    if isinstance(value, dict):
        return {key: to_builtin(item) for key, item in value.items()}
    if isinstance(value, np.generic):
        return value.item()
    return value


def format_example(text: str, annotation) -> str:
    """Render one medoid as an example block.

    The expected output is the annotation exactly as stored, with no field filtering
    beyond `note_id`, which is an internal identifier and must not be shown as part of
    the expected output.
    The layout must match few_shot.md: the only variable under test between the two
    strategies is which example is shown, not how it is presented.
    The annotation column may arrive as a JSON string or as an already deserialized dict,
    depending on how Parquet stored it.
    """
    if isinstance(annotation, (str, bytes, bytearray)):
        annotation = json.loads(annotation)

    payload = {key: to_builtin(value) for key, value in annotation.items() if key != "note_id"}
    body = json.dumps(payload, indent=2, ensure_ascii=False)
    return f"### Clinical note\n\n{text}\n\n### Expected output\n\n```json\n{body}\n```"


def l2_normalize(matrix: np.ndarray) -> np.ndarray:
    """Scale each row to unit norm, so a dot product yields the cosine similarity."""
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return matrix / norms

In [ ]:
if "dynamic" in STRATEGIES:
    medoids_df = pd.read_parquet(
        DATA_DIR / "annotations/dynamic_prompts/medoids_annotated.parquet"
    )

    # Rows of the matrix stay aligned with MEDOID_IDS by position: the argmax index is
    # what turns a similarity back into a note_id.
    MEDOID_IDS = medoids_df["note_id"].astype(str).to_numpy()
    MEDOID_MATRIX = l2_normalize(
        np.vstack([np.asarray(vector, dtype=np.float32) for vector in medoids_df["embedding"]])
    )

    # Formatting does not depend on the note, so each medoid is rendered once.
    MEDOID_EXAMPLES = {
        str(row["note_id"]): format_example(row["text"], row["annotation_json"])
        for row in medoids_df.to_dict("records")
    }

    # A note that is itself a medoid would receive its own annotation as the example.
    overlap = {str(note["note_id"]) for note in notes} & set(MEDOID_IDS)
    if overlap:
        raise RuntimeError(f"Notes present in the medoid pool: {sorted(overlap)}")

    print(f"{len(MEDOID_IDS)} medoids loaded | matrix {MEDOID_MATRIX.shape}")

In [ ]:
def dynamic_example(embedding) -> str:
    """Return the example block of the medoid nearest to a single note.

    Both sides are L2-normalised, so the dot product is the cosine similarity. On unit
    vectors argmax of the cosine and argmin of the euclidean distance select the same
    medoid, so the choice does not depend on which of the two is computed.
    """
    vector = np.asarray(embedding, dtype=np.float32).ravel()
    norm = np.linalg.norm(vector)
    if norm == 0:
        raise ValueError("Note embedding has zero norm; cosine similarity is undefined")
    vector = vector / norm

    similarities = MEDOID_MATRIX @ vector
    return MEDOID_EXAMPLES[MEDOID_IDS[int(np.argmax(similarities))]]

## Example selection per strategy

The only thing that varies across strategies is what gets injected into `{EXAMPLES}`:

- `zero_shot` passes an empty string (the placeholder simply vanishes on `.replace()`);
- `few_shot` passes the fixed pool;
- `dynamic` passes the medoid nearest to the note.

In [ ]:
def examples_for(strategy: str, note: dict) -> str:
    """Return the example block for one (strategy, note) cell."""
    if strategy == "zero_shot":
        return ""
    if strategy == "few_shot":
        return FEW_SHOT_EXAMPLES
    if strategy == "dynamic":
        return dynamic_example(note["embedding"])
    raise ValueError(f"Unknown strategy: {strategy}")

## Run directory and logging

Every fresh execution gets its own timestamped folder under `RESULTS_DIR`, so results,
the run log and the frozen configuration always travel together and no previous run is
ever overwritten:

```
<RESULTS_DIR>/<YYYYMMDD_HHMMSS>/
```

The timestamp sorts chronologically as a plain string, so no scan of existing folders is
needed to pick the next name.

On a resume, `config.json` is **not** rewritten: `run_date` has to stay the one stamped
on the records written before the interruption, otherwise notes from the same logical
run end up carrying different dates. The resume is recorded under `resumed_at` instead,
and the log is appended to rather than replaced.

In [ ]:
def setup_logging(run_dir: Path) -> logging.Logger:
    logger = logging.getLogger("extraction")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()  # avoid duplicate handlers when re-running the cell
    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"
    )
    for handler in (
        # append mode: a resumed run must not truncate the log of the interrupted one
        logging.FileHandler(run_dir / "run.log", mode="a", encoding="utf-8"),
        logging.StreamHandler(),  # to console
    ):
        handler.setFormatter(fmt)
        logger.addHandler(handler)
    return logger


run_started = datetime.now()

if RESUME_RUN:
    run_dir = RESULTS_DIR / RESUME_RUN
    config_path = run_dir / "config.json"
    if not config_path.exists():
        raise FileNotFoundError(f"No run to resume at {run_dir}")

    # The frozen config wins: run_date, environment and grid must match the records
    # already on disk, not whatever the notebook happens to hold now.
    frozen = json.loads(config_path.read_text(encoding="utf-8"))
    run_date = frozen["run_date"]
    frozen.setdefault("resumed_at", []).append(run_started.isoformat(timespec="seconds"))
    config_path.write_text(json.dumps(frozen, indent=2, ensure_ascii=False), encoding="utf-8")

    logger = setup_logging(run_dir)
    logger.info(f"Resuming run: {run_dir} | run_date={run_date}")
else:
    run_date = run_started.date().isoformat()
    run_dir = RESULTS_DIR / run_started.strftime("%Y%m%d_%H%M%S")
    run_dir.mkdir(parents=True, exist_ok=True)

    logger = setup_logging(run_dir)
    logger.info(f"Run dir: {run_dir}")

    (run_dir / "config.json").write_text(
        json.dumps(
            {
                "environment": ENVIRONMENT,
                "run_timestamp": run_started.isoformat(timespec="seconds"),
                "run_date": run_date,
                "models": models,
                "strategies": STRATEGIES,
                "n_notes": len(notes),
                "max_workers": MAX_WORKERS,
                "save_prompts": SAVE_PROMPTS,
                "ollama_config": ollama_config,
            },
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

## Preflight

`extract()` deliberately lets connection errors and timeouts propagate: if the server is
down, stopping loudly is the correct behaviour. This check fails in seconds instead of
letting the first note discover it, and confirms every model in the grid is actually
pulled before a multi-hour run starts.

Tags are normalised before comparing: `/api/tags` always reports an explicit tag, so a
model pulled without one comes back as `name:latest` and would otherwise look missing.

In [ ]:
def normalize_tag(tag: str) -> str:
    """Ollama reports untagged models as `name:latest`."""
    return tag if ":" in tag else f"{tag}:latest"


response = requests.get(f"{ollama_config['url']}/api/tags", timeout=5)
response.raise_for_status()
available = {normalize_tag(model["name"]) for model in response.json()["models"]}

missing = [model for model in models if normalize_tag(model) not in available]
if missing:
    raise RuntimeError(f"Models not pulled in Ollama: {missing}")

logger.info(f"Ollama reachable at {ollama_config['url']} | models available: {models}")

## Run the grid

Every `(model, strategy, note)` cell goes through `ExtractionRunner.extract`, which
assembles the prompt, calls Ollama in JSON mode, validates with Pydantic and writes the
record to disk. Because each cell caches its own result, rerunning this loop after an
interruption only fills in what is missing -- as long as `RESUME_RUN` points at the
interrupted directory.

`runner.load()` pays the cold load once per cell, outside the per-call `timeout_seconds`,
which is sized for generation and not for pulling a 70b into VRAM. Keeping the weights
resident across the three strategies of a model depends on `keep_alive` being set in the
Ollama block **and** being longer than a full cell; otherwise the cold load is paid
three times per model.

Failure handling belongs to `extract()`, not to this loop:

- model-side failures (`invalid`) and infrastructure failures (`error`) are **returned**
  as a record and persisted per model and per strategy under
  `<run_dir>/<model>/errors/<strategy>/<note_id>.json`, keeping the raw response;
- connection errors, timeouts, unreplaced placeholders and oversized prompts **raise**,
  cancel the queued notes and abort the run -- catching them here would silently grind
  the whole grid against a dead server, or score notes whose prompt lost its schema.

With `SAVE_PROMPTS` on, the assembled prompt of each call is written under
`<run_dir>/<model>/prompts/<strategy>/<note_id>` before the request leaves, so failed
notes keep the prompt that produced them. Cached notes are skipped whole, so a resume
does not backfill prompts for records written by an earlier run.

In [ ]:
records: list[dict] = []

for model in models:
    runner = None
    try:
        for strategy in STRATEGIES:
            runner = ExtractionRunner(
                model=model,
                strategy=strategy,
                role=role,
                template=strategy_templates[strategy],
                expected_template=expected_template,
                results_dir=run_dir,
                ollama_config=ollama_config,
                run_date=run_date,
                save_prompts=SAVE_PROMPTS,
            )

            logger.info(
                f"Start: model={model} strategy={strategy} "
                f"notes={len(notes)} workers={MAX_WORKERS}"
            )
            # Cold load once per model: a no-op after the first strategy, since
            # keep_alive holds the weights in VRAM across the whole cell.
            runner.load()

            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                futures = [
                    executor.submit(
                        runner.extract,
                        note_id=note["note_id"],
                        note_text=note["text"],
                        examples=examples_for(strategy, note),
                    )
                    for note in notes
                ]
                try:
                    for future in tqdm(
                        as_completed(futures),
                        total=len(futures),
                        desc=f"{model} | {strategy}",
                    ):
                        record = future.result()
                        records.append(record)

                        if record["status"] != "ok":
                            logger.warning(
                                f"{record['status']}: model={model} strategy={strategy} "
                                f"note_id={record['note_id']} error={record['error']}"
                            )
                        # A cut-off answer is worth its own line: the record above only
                        # says the JSON did not parse, not that it never finished.
                        usage = record.get("usage") or {}
                        if usage.get("hit_generation_cap"):
                            logger.warning(
                                f"generation cap hit: model={model} strategy={strategy} "
                                f"note_id={record['note_id']} "
                                f"prompt_tokens={usage.get('prompt_tokens')} "
                                f"completion_tokens={usage.get('completion_tokens')}"
                            )
                except BaseException:
                    # A raised failure invalidates every remaining cell: drop the
                    # queue instead of waiting for it to drain against a dead server.
                    executor.shutdown(cancel_futures=True)
                    raise

            logger.info(f"End: model={model} strategy={strategy}")
    finally:
        # Only between models: keeping the weights resident across strategies
        # is the point of keep_alive.
        if runner is not None:
            runner.unload()

## Run summary

Two tables, both written next to the results so the run is auditable without re-walking
the `errors/` tree.

**Status** -- counts per `(model, strategy)` cell:

- `ok` -- validated against the Pydantic schema;
- `invalid` -- the model answered but the output is malformed or off-schema (recall = 0);
- `error` -- infrastructure failure (non-2xx, error payload, exhausted OOM retry).

**Cost** -- tokens and latency per cell, plus two truncation counters that look alike in
the status table (both land as `invalid`) but need opposite fixes:

- `hit_generation_cap` -- notes the model stopped at `num_predict` instead of finishing.
  Read it against `prompt_tokens`: a **small** prompt with the cap hit is a degenerate
  repetition loop, and raising the cap only makes it run longer -- the fix is on the
  decoding side (a retry at a higher temperature). A prompt near `num_ctx` means the
  input genuinely leaves no room for the answer.
- `context_overflow` -- prompt and completion together filled `num_ctx`. If the prompt
  **alone** exceeded the window, Ollama truncated its START, dropping the role and the
  schema, and those scores are not interpretable; `build_prompt` now refuses that case
  up front, so a non-zero count here should mean generation ran into the ceiling.

Note that these tables cover the records held in `records`, which after a resume
includes the cached ones re-read from disk. The complete picture of the run always
lives under `run_dir`.

In [ ]:
if not records:
    raise SystemExit("No records produced -- nothing to summarise (did the grid abort?)")

runs = pd.DataFrame(records).reindex(
    columns=["model", "strategy", "note_id", "status", "error"]
)

status_summary = (
    runs.pivot_table(
        index=["model", "strategy"],
        columns="status",
        values="note_id",
        aggfunc="count",
        fill_value=0,
    )
    .reindex(columns=["ok", "invalid", "error"], fill_value=0)
    .reset_index()
)
status_summary["total"] = status_summary[["ok", "invalid", "error"]].sum(axis=1)
status_summary.to_csv(run_dir / "status_summary.csv", index=False)

# usage is None on records that never reached the server, hence the empty-dict fallback.
usage = pd.DataFrame(
    [
        {"model": record["model"], "strategy": record["strategy"], **(record["usage"] or {})}
        for record in records
    ]
).reindex(
    columns=[
        "model",
        "strategy",
        "prompt_tokens",
        "completion_tokens",
        "total_duration_s",
        "tokens_per_second",
        "hit_generation_cap",
        "context_overflow",
    ]
)

cost_summary = (
    usage.groupby(["model", "strategy"], dropna=False)
    .agg(
        prompt_tokens=("prompt_tokens", "mean"),
        completion_tokens=("completion_tokens", "mean"),
        seconds_per_note=("total_duration_s", "mean"),
        tokens_per_second=("tokens_per_second", "mean"),
        hit_generation_cap=("hit_generation_cap", "sum"),
        context_overflow=("context_overflow", "sum"),
    )
    .round(1)
    .reset_index()
)
cost_summary.to_csv(run_dir / "cost_summary.csv", index=False)

failures = runs[runs["status"] != "ok"]
failures.to_json(run_dir / "failures.json", orient="records", indent=2, force_ascii=False)

logger.info(f"Done: {len(records)} cells, {len(failures)} failed -> {run_dir}")

display(status_summary)
display(cost_summary)